# Skeleton Notebook: Modeling Stock Market Data (Practice From Scratch)

Fill in every `# TODO` cell. Each section has a short spec and, where useful,
an `assert` sanity check you can run to confirm your answer. Use
`04_Cheat_Sheet.ipynb` if you get stuck, and check `06_Solutions.ipynb` only
after attempting it yourself.

This mirrors `02_Extended_Lab.ipynb` but with the code removed — read
`00_Background_Theory.md` first if you haven't already.


## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# TODO: set pandas display options (width ~140, max_columns ~30)

# TODO: set a numpy random seed of 42 for reproducibility

plt.rcParams["figure.figsize"] = (9, 5)


## 1. Acquiring stock market data

Use the provided `data_utils.make_finviz_like_raw(n=400, seed=42)` helper to
generate a synthetic "raw" finviz-style export, save it to `finviz_raw.csv`,
then read it back with `pd.read_csv` (just like you would a real download).

In [ ]:
from data_utils import make_finviz_like_raw, simulate_price_history

# TODO: generate the raw dataframe, save to "finviz_raw.csv", then reload it into `finviz`
finviz = None  # replace

assert finviz is not None and len(finviz) == 400
finviz.head()


## 2. Summarizing the data

In [ ]:
# TODO: show the first 6 rows of just the Ticker, Company, Sector, Industry columns



In [ ]:
# TODO: print finviz.info()



In [ ]:
# TODO: use .describe(include="all") to summarize all columns (transpose it for readability)



## 3. Cleaning and exploring the data

### 3.1 Write `clean_numeric`
Write a function that takes a pandas Series of strings like `"12.34%"`,
`"$1,234.50"`, `"-"` and returns a numeric Series, treating `"-"` as missing.


In [ ]:
def clean_numeric(series: pd.Series) -> pd.Series:
    """TODO: strip %, $, and commas; convert '-' placeholders to NaN; cast to float."""
    pass

# sanity check
test = pd.Series(["12.34%", "$1,234.50", "-", "0.85"])
result = clean_numeric(test)
assert result.tolist()[:2] == [12.34, 1234.50] or True  # loosen if you round differently
print(result)


In [ ]:
def clean_market_cap(series: pd.Series) -> pd.Series:
    """TODO: convert finviz-style '1.23B' / '456.70M' strings into a $ millions float.
    Hint: B multiplies by 1000, M stays as-is."""
    pass

test_mc = pd.Series(["1.23B", "456.70M"])
print(clean_market_cap(test_mc))  # expect [1230.0, 456.7]


In [ ]:
id_cols = ["No.", "Ticker", "Company", "Sector", "Industry", "Country"]
numeric_cols = [c for c in finviz.columns if c not in id_cols + ["Market Cap"]]

# TODO: build finviz_clean by copying finviz, then applying clean_market_cap to "Market Cap"
#       and clean_numeric to every column in numeric_cols
finviz_clean = None

assert finviz_clean["Price"].dtype != object
finviz_clean.dtypes


### 3.2 Explore the price distribution

In [ ]:
# TODO: plot a histogram of finviz_clean["Price"] with 100 bins. Title it "Price Distribution (uncapped)"



In [ ]:
# TODO: plot the same histogram but only for Price < 150. What changes, and why?



### 3.3 Find and remove the outlier

In [ ]:
# TODO: compute sector_avg_prices = mean Price grouped by Sector (as a dataframe with reset_index)
sector_avg_prices = None

# TODO: bar chart sector_avg_prices with sector on x, avg price on y. Rotate x labels 90 degrees.

sector_avg_prices


In [ ]:
# TODO: find the sector with the highest average price, then within that sector find
#       the industry with the highest average price, then within that industry find the
#       single company with the highest Price. Store it in `outlier_row`.
outlier_row = None
print(outlier_row.Ticker, outlier_row.Price)


In [ ]:
# TODO: remove the outlier row from finviz_clean (filter out that Ticker), then
#       recompute and re-plot sector_avg_prices to confirm the sector average changed.



## 4. Generating relative valuations

Metrics to use: `["Price", "P/E", "PEG", "P/S", "P/B"]`


In [ ]:
metrics = ["Price", "P/E", "PEG", "P/S", "P/B"]

# TODO: compute sector_avg = mean of `metrics` grouped by Sector, columns prefixed "SAvg_"
# TODO: compute industry_avg = mean of `metrics` grouped by [Sector, Industry], prefixed "IAvg_"
# TODO: merge both onto finviz_clean (on Sector, then on [Sector, Industry]) into finviz_val
sector_avg = None
industry_avg = None
finviz_val = None

finviz_val.head()


In [ ]:
# TODO: for each metric m in `metrics`, create two 0/1 flag columns:
#   f"S_{m}_Under" = 1 if finviz_val[m] < finviz_val[f"SAvg_{m}"] else 0
#   f"I_{m}_Under" = 1 if finviz_val[m] < finviz_val[f"IAvg_{m}"] else 0
# Then sum ALL flag columns into a new column "RelValIndex" (range 0-10)
flag_cols = []


finviz_val["RelValIndex"] = None
finviz_val["RelValIndex"].describe()


In [ ]:
# TODO: filter finviz_val to RelValIndex >= 8, sort descending, and show
#       Ticker, Company, Sector, RelValIndex for the top 15 rows
potentially_undervalued = None
potentially_undervalued


## 5. Screening stocks and analyzing historical prices

### 5.1 Apply screening criteria
- Price between $20 and $100
- Volume > 10,000
- Country == "USA"
- EPS (ttm) > 0, EPS growth next year > 0, EPS growth next 5 years > 0
- Total Debt/Equity < 1
- Beta < 1.5
- Institutional Ownership < 30
- RelValIndex >= 8


In [ ]:
# TODO: build target_stocks using all the criteria above combined with & operators,
#       sorted by RelValIndex descending
target_stocks = None
print(len(target_stocks))
target_stocks[["Ticker", "Company", "RelValIndex"]]


In [ ]:
# TODO: take up to the top 6 tickers into target_tickers (list of strings).
#       If fewer than 3 pass the screen, fall back to the top 6 by RelValIndex overall.
target_tickers = None
target_tickers


### 5.2 Historical prices and moving averages

In [ ]:
stocks = simulate_price_history(target_tickers, start="2018-01-01", end="2023-12-29", seed=7)

# TODO: add MovAvg50 and MovAvg200 columns: rolling 50-day / 200-day mean of AdjClose, PER SYMBOL
#       (hint: groupby("Symbol")["AdjClose"].transform(...))


stocks.dropna().head()


In [ ]:
# TODO: for each symbol in target_tickers, plot AdjClose, MovAvg50, MovAvg200 on one chart
#       with a legend and a title like "<symbol> Daily Stock Prices"



In [ ]:
# TODO: plot ALL target tickers' AdjClose on ONE combined chart with a legend



### 5.3 Open/High/Low/Close summary

In [ ]:
# TODO: build price_summaries: for each Symbol, get open=first Open, high=max High,
#       low=min Low, close=last AdjClose
price_summaries = None
price_summaries


In [ ]:
# TODO: melt price_summaries to long form and draw one bar chart per symbol
#       (subplot grid) showing open/high/low/close



## 6. Reflection questions

1. How many stocks were flagged as the sector-price outlier, and how much did
   removing it change the Financial sector's average price?
2. How large was your final target list? What would happen if you loosened
   or tightened one criterion?
3. Which target stock had the widest gap between its 52-period high and low
   in the price_summaries table? What does that suggest about its volatility?
4. Re-read background doc §5 (Limitations) — pick one limitation and explain
   in 2-3 sentences how it could change your conclusions here.
